# Egyptian hieroglyph classifier: experiment log

Single-glyph classification (170 Gardiner classes) with a transfer-learned ResNet-18.
This notebook is the **lab notebook**: every experiment that led to the final model, in order, runnable top to bottom.
The reusable code (transforms, data, model, training, evaluation) lives in the `hieroglyphs/` package; the final
pipeline is `train.py` and the demo is `app.py`.

**How to run**
- `QUICK = True` (default): 1 epoch and 1 seed per experiment, about 15 minutes for the whole notebook. It simulates the experiment end to end; the numbers will not match the recorded ones.
- `QUICK = False`: the real experiment (10 epochs, 3 seeds). One epoch takes ~100 s, so expect **several hours**.
- Everything is written to `runs/notebook/`. The canonical model in `weights/` is **never** touched by this notebook; rebuild it with `python train.py --overwrite`.

Blockquotes marked **Recorded** are results from the original runs (Day 90-91).

In [ ]:
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Subset
from torchvision import transforms

from hieroglyphs import config
from hieroglyphs.data import HieroglyphTestDataset, dev_split, make_loader, train_folder
from hieroglyphs.evaluate import evaluate
from hieroglyphs.model import build_model, load_weights
from hieroglyphs.predict import predict_topk
from hieroglyphs.labels import load_labels
from hieroglyphs.training import save_checkpoint, train_model
from hieroglyphs.transforms import test_transform, train_transform

QUICK = True  # True: ~15 min simulation. False: the real experiment (hours).
EPOCHS = 1 if QUICK else config.EPOCHS
SEEDS = [0] if QUICK else [0, 1, 2]
OUT_DIR = config.RUNS_DIR / "notebook"  # scratch space, never the canonical model
torch.manual_seed(0)
print(f"QUICK={QUICK}  epochs={EPOCHS}  seeds={SEEDS}  outputs -> {OUT_DIR}")

## 1. Data

- **Train:** one folder per class (`ImageFolder`), 3,584 images, 170 classes. Many classes have very few images (40 have exactly one).
- **Test:** a flat folder with the label in the filename (`070048_F34.png`), so it needs a custom `Dataset`. `UNKNOWN` and `M20` do not exist in training, so they are filtered out: **silent label mismatches are worse than a crash**.
- Every image is **50 px wide x 75 px tall**, so no resizing is needed anywhere.

> **Recorded:** train 3584 / test 444 / 170 classes / tensor shape `[3, 75, 50]` / 11 test files removed (`UNKNOWN`, `M20`).

In [ ]:
train_dataset = train_folder(train_transform)        # augmented: used for training
train_eval_dataset = train_folder(test_transform)    # clean view of the same images: used to MEASURE train accuracy
assert train_eval_dataset.class_to_idx == train_dataset.class_to_idx

test_dataset = HieroglyphTestDataset(train_dataset.class_to_idx, transform=test_transform)
test_loader = make_loader(test_dataset)
full_train_eval_loader = make_loader(train_eval_dataset)

removed = [f.name for f in config.TEST_DIR.glob("*.png")
           if HieroglyphTestDataset.label_of(f.name) not in train_dataset.class_to_idx]
print(len(train_dataset), len(test_dataset), len(train_dataset.classes), test_dataset[0][0].shape)
print(len(removed), {HieroglyphTestDataset.label_of(f) for f in removed})

## 2. Baseline: transfer learning

ResNet-18 pretrained on ImageNet (ImageNet weights transfer because early layers are generic edge/texture detectors; MNIST weights would not).
`build_model()` does the following:
1. freeze the whole backbone (`requires_grad = False`),
2. unfreeze only `layer4` (frozen-only gave ~57% test, an under-fitting / high-bias result),
3. replace `fc` with a fresh `Linear(512, 170)` (a new layer is trainable by default),

and `train_model()` trains it with Adam using two parameter groups: head `lr=1e-3`, `layer4` `lr=1e-4` (10x smaller for pretrained layers).

> **Recorded (Day 90), baseline with weight decay 0.01 on layer4:** train 0.9523 / test 0.8018.

In [ ]:
trainable = sum(p.numel() for p in build_model(pretrained=False).parameters() if p.requires_grad)
print(f"trainable parameters: {trainable:,}")

model, classes = train_model(epochs=EPOCHS, weight_decay=0.01, verbose=True)
print(f"baseline: train={evaluate(model, full_train_eval_loader):.4f}  test={evaluate(model, test_loader):.4f}")

## 3. Measuring honestly

Two silent bugs made every early accuracy number untrustworthy, and both are now impossible by construction:
1. **`model.eval()` was missing during evaluation**, so BatchNorm kept normalizing with per-batch statistics (and updating its running stats). `evaluate()` now always switches to eval mode.
2. **"Train accuracy" was measured on augmented images** through the training loader. It is now measured on a clean (non-augmented) view of the data.

To stop choosing configurations by repeatedly looking at the test set, hold out a **stratified 20% dev split** of the training data (per class; classes with one image stay in train, because sklearn's `stratify` raises an error there). Experiments below compare configurations on **dev**; the test set is touched once, for the final model.

In [ ]:
train_idx, dev_idx = dev_split(train_dataset.targets)
dev_loader = make_loader(Subset(train_eval_dataset, dev_idx))
subtrain_eval_loader = make_loader(Subset(train_eval_dataset, train_idx))
print(len(train_idx), "train /", len(dev_idx), "dev")


def run_experiment(seed, transform, weight_decay=0.0, epochs=EPOCHS,
                   lr_fc=config.LR_FC, lr_layer4=config.LR_LAYER4):
    """Train on the 80% split; return (train accuracy, dev accuracy), both on clean images."""
    model, _ = train_model(seed, transform, weight_decay, epochs, lr_fc, lr_layer4, subset_idx=train_idx)
    return evaluate(model, subtrain_eval_loader), evaluate(model, dev_loader)

## 4. Does augmentation help? Simple vs gentle

- Flips are **rejected** on purpose: sign orientation can be meaningful.
- `simple`: rotation 5 + brightness. `gentle`: brightness + small affine (rotate 5, translate 3%, scale 0.95-1.05) + autocontrast.
- Single-variable changes one at a time (a typo such as `scale=(0.59, 1.05)` once produced the worst result of the day, which is why `run_experiment` builds everything fresh and seeded).

> **Recorded, dev split (the original cell printed this as "test"; it was dev), seeds 0/1/2:**
>
> | | seed 0 | seed 1 | seed 2 |
> |---|---|---|---|
> | simple | 0.8492 | 0.8421 | 0.8677 |
> | gentle | 0.8606 | 0.8762 | 0.8563 |
>
> About one point apart against a seed-to-seed spread of 1.3-5.9 points: **no real difference, within noise.**

In [ ]:
simple_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
])

results = {}
for name, tf in [("simple", simple_transform), ("gentle", train_transform)]:
    results[name] = [run_experiment(seed, tf) for seed in SEEDS]
    for seed, (tr, dev) in zip(SEEDS, results[name]):
        print(f"{name} seed={seed}: train={tr:.4f} dev={dev:.4f}")

for name, runs in results.items():
    devs = [dev for _, dev in runs]
    print(f"{name}: mean dev={np.mean(devs):.4f}  seed-to-seed spread={np.ptp(devs):.4f}")

## 5. Photometric augmentation (a dead end)

Hypothesis: the train/test gap comes from lighting differences between manuscripts, so augment contrast/blur.
Result: train accuracy jumped to ~99% (near-total memorization) while dev did not improve, so it is a *weaker* regularizer than the gentle one.

> **Recorded (dev):** train 0.9927 / 0.9917 / 0.9875 (avg 0.9906), dev 0.8691 / 0.8606 / 0.8492 (avg 0.8596). **Ruled out.**

In [ ]:
photometric_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomAutocontrast(p=0.2),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
])

photometric = [run_experiment(seed, photometric_transform) for seed in SEEDS]
for seed, (tr, dev) in zip(SEEDS, photometric):
    print(f"photometric seed={seed}: train={tr:.4f} dev={dev:.4f}")
print(f"train avg={np.mean([r[0] for r in photometric]):.4f}; dev avg={np.mean([r[1] for r in photometric]):.4f}")

## 6. Weight decay

Tested 0 / 1e-4 / 1e-2 on the `layer4` group. **No meaningful effect at any magnitude.** The reason: with BatchNorm after every conv, the network is invariant to the *scale* of the conv weights, so an L2 penalty that only shrinks scale has nothing to push against.

In [ ]:
for wd in [0.0, 1e-4, 1e-2]:
    tr, dev = run_experiment(0, train_transform, weight_decay=wd)
    print(f"weight_decay={wd:g}: train={tr:.4f} dev={dev:.4f}")

## 7. Final model

Locked config (chosen on dev, **not** cherry-picked on test): gentle augmentation, full training data, seed 0, 10 epochs, no weight decay.
First a smoke test of the save/evaluate path with `epochs=0` (an untrained head should score ~0%), then the real run.

> **Recorded final:** train 0.9509 / test 0.7928 (a ~16-point gap: genuine high variance).
> **Finding:** dev (carved from train) gives ~86% but test gives ~79%. This ~7-point gap is a real **train/test distribution difference** (different scans, manuscripts, lighting), i.e. the same *train-serving skew* idea as in the MNIST project, not a modeling bug.

In [ ]:
smoke, smoke_classes = train_model(epochs=0)
save_checkpoint(smoke, smoke_classes, OUT_DIR / "smoke")
print("smoke test (untrained head), expect ~0:", evaluate(smoke, test_loader))

In [ ]:
final, final_classes = train_model(seed=0, transform=train_transform, epochs=EPOCHS, verbose=True)
save_checkpoint(final, final_classes, OUT_DIR / "final")  # saved before evaluating
print(f"final: train={evaluate(final, full_train_eval_loader):.4f}  test={evaluate(final, test_loader):.4f}")

## 8. The bug that motivated the refactor: preprocessing drift

The Streamlit app had its own copy of the preprocessing, with `Resize((50, 75))`. `Resize` reads sizes as **(height, width)**, but the images are 50 wide x 75 tall, so the app squashed every tall glyph into a short, wide one.
For `070048_F34.png` the notebook said **F34 (78.0%)** and the app said **I9 (16.4%)**. Fix: one definition of the transforms in `hieroglyphs/`, and tests that guard it (`tests/test_pipeline.py`).
Below: the canonical saved model, evaluated both ways (read-only).

In [ ]:
canonical = load_weights(build_model(pretrained=False), config.WEIGHTS_DIR / "weights.pt")
labels = load_labels()
sample = Image.open(config.TEST_DIR / "070048_F34.png")
print("image size (width, height):", sample.size)

print("correct path (no resize):  ", predict_topk(canonical, labels, sample, k=2))

wrong = transforms.Compose([transforms.Resize((50, 75)), test_transform])  # the old app's preprocessing
with torch.no_grad():
    probs = torch.softmax(canonical(wrong(sample.convert("RGB")).unsqueeze(0)), dim=1)[0]
top = probs.topk(2)
print("old app (Resize((50, 75))):", [(labels[i], round(p, 3)) for p, i in zip(top.values.tolist(), top.indices.tolist())])

## Summary of what was learned

| Question | Answer |
|---|---|
| Frozen backbone only? | High bias (~57% test). Unfreeze `layer4`. |
| More capacity (`layer3`)? | Train up, test flat: an early variance signal. Reverted. |
| Augmentation (simple vs gentle)? | Within seed noise. |
| Photometric augmentation? | Memorizes (train ~99%), dev no better. Ruled out. |
| Weight decay? | No effect (BatchNorm scale invariance). |
| Why is test (79%) below dev (86%)? | Real train/test distribution gap in the dataset. |
| Why did the app disagree with the notebook? | Duplicated preprocessing with swapped (H, W). |

**Next:** a lookup/description feature (lexicon or Unikemet) in the app, then a README documenting this whole process, bugs and dead ends included.